<a href="https://colab.research.google.com/github/uixPhuke/AI_and_Machine_Learning/blob/main/footballllll.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
folder='/content/drive/MyDrive/BigData'
print(os.listdir(folder))

['Academic_Performance.xlsx', 'Day1.ipynb', 'Placement.csv', 'Academic_Performance.csv', 'Placemen1.csv', 'transfers.csv']


In [ ]:
from pyspark.sql import SparkSession

# a. Create SparkSession
spark = SparkSession.builder \
    .appName("TransferDatasetAnalysis") \
    .master("local[*]") \
    .getOrCreate()

# b. Load CSV dataset
df = spark.read.csv(
    "/content/drive/MyDrive/BigData/transfers.csv",
    header=True,
    inferSchema=True
)

# c. Display first 10 records
df.show(10, truncate=False)

# d. Display schema
df.printSchema()

# e. Count total number of records
print("Total number of records:", df.count())

+---------+-------------+---------------+------------+----------+---------------+------------+------------+-------------------+-----------------+
|player_id|transfer_date|transfer_season|from_club_id|to_club_id|from_club_name |to_club_name|transfer_fee|market_value_in_eur|player_name      |
+---------+-------------+---------------+------------+----------+---------------+------------+------------+-------------------+-----------------+
|467994   |2030-06-30   |25/26          |5621        |749       |Reggiana       |FC Empoli   |0.0         |700000.0           |Luca Belardinelli|
|645842   |2028-02-02   |27/28          |6505        |19684     |Gimcheon Sangmu|Jeju SK     |0.0         |150000.0           |Chan-gi An       |
|677470   |2028-02-02   |27/28          |6505        |3535      |Gimcheon Sangmu|Ulsan HD    |0.0         |400000.0           |Yool Heo         |
|709155   |2028-02-02   |27/28          |6505        |19684     |Gimcheon Sangmu|Jeju SK     |0.0         |325000.0         

In [ ]:
#2A
from pyspark.sql.functions import col, sum

null_counts = df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

null_counts.show()

+---------+-------------+---------------+------------+----------+--------------+------------+------------+-------------------+-----------+
|player_id|transfer_date|transfer_season|from_club_id|to_club_id|from_club_name|to_club_name|transfer_fee|market_value_in_eur|player_name|
+---------+-------------+---------------+------------+----------+--------------+------------+------------+-------------------+-----------+
|        0|            0|              0|           0|         0|             0|           0|       61526|              68160|          0|
+---------+-------------+---------------+------------+----------+--------------+------------+------------+-------------------+-----------+



In [ ]:
#2B
df_clean = df.dropna()

In [ ]:
print("Records after removing NULL values:", df_clean.count())

Records after removing NULL values: 87137


In [ ]:
df_clean = df.fillna({
    "transfer_fee": 0,
    "market_value_in_eur": 0,
    "from_club_name": "Unknown",
    "to_club_name": "Unknown"
})

In [ ]:
#2C
df_clean = df_clean.dropDuplicates()

In [ ]:
print("Records after removing duplicates:", df_clean.count())

Records after removing duplicates: 87137


In [ ]:
#2D
df_clean = df_clean.withColumnRenamed(
    "player_name",
    "PlayerName"
)

In [ ]:
df_clean.printSchema()

root
 |-- player_id: integer (nullable = true)
 |-- transfer_date: date (nullable = true)
 |-- transfer_season: string (nullable = true)
 |-- from_club_id: integer (nullable = true)
 |-- to_club_id: integer (nullable = true)
 |-- from_club_name: string (nullable = true)
 |-- to_club_name: string (nullable = true)
 |-- transfer_fee: double (nullable = true)
 |-- market_value_in_eur: double (nullable = true)
 |-- PlayerName: string (nullable = true)



In [ ]:
#2E
df_clean.show(10, truncate=False)

+---------+-------------+---------------+------------+----------+---------------+---------------+------------+-------------------+--------------------+
|player_id|transfer_date|transfer_season|from_club_id|to_club_id|from_club_name |to_club_name   |transfer_fee|market_value_in_eur|PlayerName          |
+---------+-------------+---------------+------------+----------+---------------+---------------+------------+-------------------+--------------------+
|746780   |2026-08-03   |26/27          |2996        |6505      |Incheon Utd.   |Gimcheon Sangmu|0.0         |125000.0           |Young-hun Kang      |
|322873   |2026-06-30   |25/26          |152         |36        |Samsunspor     |Fenerbahçe     |0.0         |1200000.0          |İrfan Can Eğribayat |
|374720   |2026-06-30   |25/26          |18659       |18577     |AD San Carlos  |Cartaginés     |0.0         |200000.0           |Christian Martínez  |
|393745   |2026-06-30   |25/26          |122         |3         |Sturm Graz     |1.FC Kö

In [ ]:
from pyspark.sql.functions import count, avg, max, min, col

# Select categorical column: from_club_name
# Select numerical column: transfer_fee

result = df_clean.groupBy("from_club_name").agg(
    count("*").alias("total_records"),
    avg("transfer_fee").alias("average_transfer_fee"),
    max("transfer_fee").alias("maximum_transfer_fee"),
    min("transfer_fee").alias("minimum_transfer_fee")
)

# Display aggregation result
print("Aggregation Result:")
result.show(20, truncate=False)

# Sort by count in descending order
result_sorted = result.orderBy(
    col("total_records").desc()
)

print("Sorted Result:")
result_sorted.show(20, truncate=False)

Aggregation Result:
+---------------+-------------+--------------------+--------------------+--------------------+
|from_club_name |total_records|average_transfer_fee|maximum_transfer_fee|minimum_transfer_fee|
+---------------+-------------+--------------------+--------------------+--------------------+
|R Charleroi SC |84           |1149464.2857142857  |2.24E7              |0.0                 |
|Coritiba FC    |100          |341500.0            |6000000.0           |0.0                 |
|Mainz          |105          |2029523.8095238095  |3.15E7              |0.0                 |
|Palermo        |35           |173714.2857142857   |2650000.0           |0.0                 |
|FC Imabari     |7            |0.0                 |0.0                 |0.0                 |
|FK Minsk       |17           |25294.117647058825  |180000.0            |0.0                 |
|UE Olot        |2            |0.0                 |0.0                 |0.0                 |
|Puntarenas FC  |3            

In [ ]:
from pyspark.sql.functions import col, sum

# a. Identify NULL values
null_counts = df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

print("NULL values:")
null_counts.show()

# b. Remove NULL values
df_clean = df.dropna()

# c. Remove duplicate records
df_clean = df_clean.dropDuplicates()

# d. Rename a column
df_clean = df_clean.withColumnRenamed(
    "player_name",
    "PlayerName"
)

# e. Display transformed dataset
print("Transformed Dataset:")
df_clean.show(10, truncate=False)